In [1]:
import sys

sys.path.append("..")
from transformers import AutoTokenizer, AutoProcessor, CLIPModel
import torch
from torch.nn import functional as F
from src.utils import get_device, read_index
import json
import polars as pl
from src.geoclip import GeoCLIP
from PIL import Image
import os
from tqdm import tqdm
import numpy as np
from src.metrics import evaluate

device = "cuda" if torch.cuda.is_available() else "cpu"
clip_model = "openai/clip-vit-large-patch14"
topk = 200
base_img_path = "../../datasets/google-landmark/index-img"
batch_size = 256

In [2]:
model = CLIPModel.from_pretrained(clip_model).to(device)
processor = AutoProcessor.from_pretrained(clip_model)
tokenizer = AutoTokenizer.from_pretrained(clip_model)
geoclip_model = GeoCLIP().to(device)

Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-large-patch14
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-large-patch14
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
df_index = pl.read_csv(
    "../../datasets/google-landmark/index_ref_predicted_siglip_filtered.csv"
)
test_set = json.loads(open("../../datasets/geocir-triplet/test.v1.json", "r").read())[
    "data"
]

In [4]:
index, meta = read_index("../index/clip_index_gld_siglip_filtered")
meta = meta["metadata"]
index.ntotal

203041

In [41]:
query = "A castle landmark in Morocco"
query_tok = tokenizer(query, truncation=True, padding="max_length", max_length=77, return_tensors="pt")
with torch.no_grad():
    out = model.get_text_features(**{k: v.to(device) for k, v in query_tok.items()}).pooler_output.cpu()
    out = F.normalize(out, dim=-1).numpy()

sim, ind = index.search(out, 100)

In [42]:
df_index[ind[0]]

row_idx,landmark_id,input_url,resolved_url,geohack_url,latitude,longitude,error,supercategory,hierarchical_label,natural_or_human_made,rg_name,rg_admin1,rg_admin2,country_code,country,continent,subregion,id,category,confidence
i64,i64,str,str,str,f64,f64,str,str,str,str,str,str,str,str,str,str,str,str,str,f64
134828,98539,"""http://commons.wikimedia.org/w…","""https://commons.wikimedia.org/…","""https://geohack.toolforge.org/…",31.061944,-7.916111,null,"""mountain range""","""mountain""","""natural""","""Iguidi""","""Souss-Massa-Draa""","""Taroudannt""","""MA""","""Morocco""","""Africa""","""Northern Africa""","""1c0093e02f515701""","""castle_fortress""",0.06796
164695,98539,"""http://commons.wikimedia.org/w…","""https://commons.wikimedia.org/…","""https://geohack.toolforge.org/…",31.061944,-7.916111,null,"""mountain range""","""mountain""","""natural""","""Iguidi""","""Souss-Massa-Draa""","""Taroudannt""","""MA""","""Morocco""","""Africa""","""Northern Africa""","""ece8427ff9d02ef2""","""palace_manor""",0.055173
137050,60124,"""http://commons.wikimedia.org/w…","""https://commons.wikimedia.org/…","""https://geohack.toolforge.org/…",39.4793,-0.375933,null,"""fortified tower""","""castle / fort""","""human-made""","""Valencia""","""Valencia""","""Provincia de Valencia""","""ES""","""Spain""","""Europe""","""Southern Europe""","""c15a6c92a6f43185""","""arch_gate""",0.582877
67870,97420,"""http://commons.wikimedia.org/w…","""https://commons.wikimedia.org/…","""https://geohack.toolforge.org/…",-25.257185,16.541341,null,"""fortress""","""castle / fort""","""human-made""","""Maltahohe""","""Hardap""","""""","""NA""","""Namibia""","""Africa""","""Southern Africa""","""9e5bb6f8e7c17e89""","""castle_fortress""",0.162762
71825,91887,"""http://commons.wikimedia.org/w…","""https://commons.wikimedia.org/…","""https://geohack.toolforge.org/…",29.550668,34.966561,null,"""park""","""parks""","""natural""","""Eilat""","""Southern District""","""""","""IL""","""Israel""","""Asia""","""Western Asia""","""4e93ac921e0c92d6""","""palace_manor""",0.222971
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
86222,87357,"""http://commons.wikimedia.org/w…","""https://commons.wikimedia.org/…","""https://geohack.toolforge.org/…",42.446454,2.944778,null,"""castle""","""castle / fort""","""human-made""","""Cantallops""","""Catalonia""","""Provincia de Girona""","""ES""","""Spain""","""Europe""","""Southern Europe""","""7c5c5af72d0b5b04""","""castle_fortress""",0.131177
31349,13482,"""http://commons.wikimedia.org/w…","""https://commons.wikimedia.org/…","""https://geohack.toolforge.org/…",30.984851,-8.228336,null,"""mosque""","""mosque""","""human-made""","""Azgour""","""Marrakech-Tensift-Al Haouz""","""Al-Haouz""","""MA""","""Morocco""","""Africa""","""Northern Africa""","""68f051695317f121""","""arch_gate""",0.066109
18370,62261,"""http://commons.wikimedia.org/w…","""https://commons.wikimedia.org/…","""https://geohack.toolforge.org/…",36.604722,47.234167,null,"""archaeological site""","""archeological site""","""human-made""","""Takab""","""Azarbayjan-e Gharbi""","""""","""IR""","""Iran""","""Asia""","""Southern Asia""","""4f699da1121cc27e""","""arch_gate""",0.440288


In [ ]:
all_gt = [b["target_img_ids"] for b in test_set]
query_img_ids = [b["ref_img_id"] for b in test_set]
all_pred = []

a, b = 0.8, 0.2

for i in tqdm(range(0, len(test_set), batch_size), desc="batch test"):
    batch = test_set[i : i + batch_size]
    img_inputs = processor(
        [Image.open(os.path.join(base_img_path, f"{b['img_id']}.jpg")) for b in batch],
        return_tensors="pt",
    )
    captions = geoclip_model.text_encoder.preprocess_text([b["caption"] for b in batch])

    with torch.no_grad():
        # 1st stage - batch image retrieval
        img_query_emb = model.get_image_features(
            img_inputs["pixel_values"].to(device)
        ).pooler_output.cpu()
        img_query_emb = F.normalize(img_query_emb, dim=-1)

        sim1, ind1 = index.search(img_query_emb, topk)  # (B, topk) each

        # 2nd stage - batch text embeddings
        txt_query_emb = geoclip_model.text_encoder(
            **{k: v.to(device) for k, v in captions.items()}
        )
        txt_query_emb = F.normalize(txt_query_emb, dim=-1).cpu()  # (B, dim)

        # per-item reranking (candidates differ per item)
        for j in range(len(batch)):
            df_top_j = df_index[ind1[j]][
                ["row_idx", "id", "landmark_id", "category", "country", "latitude", "longitude"]
            ]
            coords_j = (
                torch.tensor(df_top_j[["latitude", "longitude"]].to_numpy())
                .float()
                .to(device)
            )

            loc_emb_j = geoclip_model.location_encoder(coords_j)
            loc_emb_j = F.normalize(loc_emb_j, dim=-1).cpu()  # (topk, dim)

            sim2_j = (txt_query_emb[j : j + 1] @ loc_emb_j.t()).view(-1)  # (topk,)

            sim1_norm_j = F.normalize(torch.tensor(sim1[j]), dim=-1)
            sim2_norm_j = F.normalize(sim2_j, dim=-1)
            final_sim_j = a * sim1_norm_j + b * sim2_norm_j
            rank_j = torch.argsort(final_sim_j, descending=True).tolist()

            all_pred.append(df_top_j["row_idx"][rank_j].to_list())

evaluate(all_pred, all_gt, query_img_ids)

batch test: 100%|██████████| 64/64 [03:47<00:00,  3.55s/it]


{'mAP@5': 0.11350952507803802,
 'mAP@10': 0.10804345303553824,
 'mAP@25': 0.10566761721467133,
 'mAP@50': 0.10519833569476147,
 'mAP@100': 0.10516384093399622}